In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import yahooquery as yq
import sys
sys.path.append("../")
from src import calculations,data_cleaning,data_import,name_to_Ticker,plots,data_cleaning,screening


ModuleNotFoundError: No module named 'calculations'

Close
Daily_Return
Cumulative_Return
SMA_20
SMA_50
SMA_200
EMA_20
EMA_50
EMA_200
RSI_14
ATR_14
MACD
MACD_Signal
MACD_Histogram
OBV

In [ ]:
df=yf.Ticker("MSFT").history(period="1y")
df=data_cleaning.data_clean(df)
price_metrics=calculations.calculate_price_metrics(df)
moving_averages=calculations.calculate_moving_averages(df)
rsi=calculations.calc_rsi(df)
atr=calculations.calculate_atr(df)
macd=calculations.calculate_macd(df)
volume=calculations.volume_metrics(df)

In [ ]:
calculated_df=pd.DataFrame()
calculated_df=pd.concat([df["Close"],price_metrics["daily_return"],price_metrics["cumulative_return"],moving_averages,rsi,atr,macd,volume["OBV"]],axis=1)
calculated_df

In [ ]:
calculated_df.columns

In [ ]:
latest=calculated_df.iloc[-1]

In [ ]:
latest["RSI"]<30

In [ ]:
latest["Close"] > latest["SMA_200"]

In [ ]:
latest["MACD"] > latest["MACD_SIGNAL"]

In [ ]:
latest["OBV"] > 0

In [ ]:
(
    (latest["RSI"] < 30)
    & (latest["Close"] > latest["SMA_200"])
    & (latest["MACD"] > latest["MACD_SIGNAL"])
)

In [ ]:
def apply_condition(df, metric, operator, value):

    latest = df.iloc[-1]
    if isinstance(value, str):
        value = latest[value]

    if operator == ">":
        return latest[metric] > value

    elif operator == "<":
        return latest[metric] < value

    elif operator == ">=":
        return latest[metric] >= value

    elif operator == "<=":
        return latest[metric] <= value

    elif operator == "==":
        return latest[metric] == value

    elif operator == "!=":
        return latest[metric] != value

    else:
        raise ValueError("Invalid operator")

def screen_stock(df, conditions):

    for metric, operator, value in conditions:

        if not apply_condition(df, metric, operator, value):
            return False

    return True
condition=[("RSI","<",30),("RSI",">","SMA20")]
screen_stock(calculated_df,condition)

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN"]
conditions = [
    ("RSI", "<", 30),
    ("Close", ">", "SMA_200"),
    ("MACD", ">", "MACD_SIGNAL")
]
def screen_stocks(tickers, conditions):
    results={}
    for ticker in tickers:
        df=data_import.load_stock_data(ticker)
        df=data_cleaning.data_clean(df)
        calculated_df=calculations.calculate_all(df)
        if screening.screen_stock(calculated_df,conditions):
            results[ticker]={"df":df,"calculated":calculated_df}
    return results
screen_stocks(tickers,conditions)

In [ ]:
url="https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500=pd.read_html(url)[0]
tickers=sp500["Symbol"].tolist()
